In [1]:

from typing import Dict, Tuple
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.datasets import MNIST
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

%matplotlib inline

from torch import autograd
from torch.autograd import Variable
from tensorboardX import SummaryWriter
import torch.optim as optim
import torchvision.datasets as datasets
import time
import os


if __name__ == "__main__":
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.device_count() > 0 else "No GPU detected")

Torch version: 2.6.0
CUDA available: False
CUDA version: None
Number of GPUs: 0
GPU name: No GPU detected


In [2]:
def plot_shape(shape_matrix):
    """Plot the generated shape (expects input shape (1, 32, 32) or (32, 32))."""
    # Squeeze channel if present
    if shape_matrix.ndim == 3 and shape_matrix.shape[0] == 1:
        shape_matrix = shape_matrix.squeeze(0)  # → (32, 32)

    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
    ax.set_facecolor('#301934')
    ax.imshow(shape_matrix, origin='upper', cmap='viridis')  # add colormap if needed
    plt.axis('off')
    # print(f'size: {shape_matrix.shape[0]} x {shape_matrix.shape[1]}')
    plt.show()


def load_item(item, p= True, action=''):
    if action=='':
        if p:
            print(f'Cond: {item[0]}')
            print(f'Params: {item[1]}')
        plot_shape(item[2])
        return {'Cond':item[0], 'Params':item[1]}
    if action == 'shape':
        return item[3]
    
def quarter(matrix):
    return matrix[:32, :32]

In [3]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.q = nn.Conv2d(in_channels, in_channels, 1)
        self.k = nn.Conv2d(in_channels, in_channels, 1)
        self.v = nn.Conv2d(in_channels, in_channels, 1)
        self.proj = nn.Conv2d(in_channels, in_channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q(x).reshape(B, C, -1)
        k = self.k(x).reshape(B, C, -1)
        v = self.v(x).reshape(B, C, -1)
        attn = torch.softmax(q.transpose(1,2) @ k / (C**0.5), dim=-1)
        out = (attn @ v.transpose(1,2)).transpose(1,2).reshape(B, C, H, W)
        return self.proj(out) + x

In [4]:
class AdaIN(nn.Module):
    def __init__(self, channels, cond_dim):
        super().__init__()
        self.fc = nn.Linear(cond_dim, channels*2)
    def forward(self, x, cond):
        h = self.fc(cond)
        gamma, beta = h.chunk(2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta = beta.unsqueeze(-1).unsqueeze(-1)
        mean = x.mean([2,3], keepdim=True)
        std = x.std([2,3], keepdim=True)
        x_norm = (x - mean) / (std + 1e-5)
        return gamma * x_norm + beta

In [21]:
def get_timestep_embedding(timesteps, embedding_dim):
    """
    timesteps: 1-D  (B,)  or 2-D (B,1) tensor of integers / floats
    returns:   (B, embedding_dim) sinusoidal embedding
    """
    if timesteps.ndim == 2:
        timesteps = timesteps.squeeze(-1)          # (B,)
    assert timesteps.ndim == 1                     # ensure 1-D
    half_dim = embedding_dim // 2
    exponents = torch.arange(half_dim, device=timesteps.device) / half_dim
    freqs = 10000 ** (-exponents)
    angles = timesteps.float()[:, None] * freqs[None, :]  # (B, half_dim)
    emb = torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)
    return emb                                    # (B, embedding_dim)


In [6]:
class ImprovedResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, cond_dim=8, use_attention=False):
        super().__init__()
        self.same_channels = in_channels == out_channels
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.ada = AdaIN(out_channels, cond_dim)
        self.use_attention = use_attention
        if use_attention:
            self.attn = SelfAttention(out_channels)
        else:
            self.attn = nn.Identity()
    def forward(self, x, cond):
        h = F.gelu(self.norm1(self.conv1(x)))
        h = self.ada(h, cond)
        h = F.gelu(self.norm2(self.conv2(h)))
        h = self.attn(h)
        if self.same_channels:
            return (x + h) / 1.414
        else:
            return h

In [29]:
class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=1, base=64, cond_dim=8, time_dim=128):
        super().__init__()
        self.time_dim = time_dim
        self.time_embed = nn.Linear(time_dim, cond_dim)
        # Down
        self.enc1 = ImprovedResBlock(in_channels, base, cond_dim, use_attention=False)
        self.enc2 = ImprovedResBlock(base, base*2, cond_dim, use_attention=True)
        self.enc3 = ImprovedResBlock(base*2, base*4, cond_dim, use_attention=True)
        self.enc4 = ImprovedResBlock(base*4, base*8, cond_dim, use_attention=True)
        # Up
        self.up1 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.dec1 = ImprovedResBlock(base*8, base*4, cond_dim, use_attention=True)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.dec2 = ImprovedResBlock(base*4, base*2, cond_dim, use_attention=True)
        self.up3 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.dec3 = ImprovedResBlock(base*2, base, cond_dim, use_attention=False)
        self.out = nn.Conv2d(base, in_channels, 1)
    def forward(self, x, cond, t, context_mask=0):
        cond = cond * (1-context_mask)
        t_emb = get_timestep_embedding(t, self.time_dim).to(x.device)
        cond = cond + self.time_embed(t_emb)
        e1 = self.enc1(x, cond)
        e2 = self.enc2(F.avg_pool2d(e1, 2), cond)
        e3 = self.enc3(F.avg_pool2d(e2, 2), cond)
        e4 = self.enc4(F.avg_pool2d(e3, 2), cond)
        d1 = self.up1(e4)
        d1 = torch.cat([d1, e3], 1)
        d1 = self.dec1(d1, cond)
        d2 = self.up2(d1)
        d2 = torch.cat([d2, e2], 1)
        d2 = self.dec2(d2, cond)
        d3 = self.up3(d2)
        d3 = torch.cat([d3, e1], 1)
        d3 = self.dec3(d3, cond)
        return self.out(d3)

In [22]:
import torch
import torch.nn as nn
from collections import OrderedDict

# ---------- import or paste your ImprovedUNet definition here ----------
# from model_zoo import ImprovedUNet

# ---- CONFIG ----
BATCH = 4          # test batch-size
IN_CH = 1          # model input channels
BASE   = 64        # same base you used when instantiating ImprovedUNet
COND   = 8
TIME   = 128
IMG_H  = IMG_W = 32

# ---- register hooks to capture tensor shapes ----
def make_shape_hook(name, store):
    def _hook(_, __, output):
        store[name] = tuple(output.shape)
    return _hook

def check_improved_unet_shapes():
    model = ImprovedUNet(in_channels=IN_CH,
                         base=BASE,
                         cond_dim=COND,
                         time_dim=TIME)

    shape_log = OrderedDict()
    # Attach hooks to every sub-module we care about
    for tag, block in [
        ("enc1", model.enc1),
        ("enc2", model.enc2),
        ("enc3", model.enc3),
        ("enc4", model.enc4),
        ("up1",  model.up1),
        ("dec1", model.dec1),
        ("up2",  model.up2),
        ("dec2", model.dec2),
        ("up3",  model.up3),
        ("dec3", model.dec3),
        ("out" , model.out),
    ]:
        block.register_forward_hook(make_shape_hook(tag, shape_log))

    # ----- dummy data -----
    x   = torch.zeros(BATCH, IN_CH, IMG_H, IMG_W)
    c   = torch.zeros(BATCH, COND)
    t   = torch.rand  (BATCH, 1)

    out = model(x, c, t)

    # ----- assertions -----
    assert out.shape == (BATCH, IN_CH, IMG_H, IMG_W), \
        f"Final output {out.shape} != expected {(BATCH, IN_CH, IMG_H, IMG_W)}"

    # Optional: verify a few key internal dimensions
    expected = {
        "enc1": (BATCH, BASE,      IMG_H,       IMG_W),
        "enc2": (BATCH, BASE*2,    IMG_H//2,    IMG_W//2),
        "enc3": (BATCH, BASE*4,    IMG_H//4,    IMG_W//4),
        "enc4": (BATCH, BASE*8,    IMG_H//8,    IMG_W//8),
        "up1":  (BATCH, BASE*4,    IMG_H//4,    IMG_W//4),
        "dec1": (BATCH, BASE*4,    IMG_H//4,    IMG_W//4),
        "up2":  (BATCH, BASE*2,    IMG_H//2,    IMG_W//2),
        "dec2": (BATCH, BASE*2,    IMG_H//2,    IMG_W//2),
        "up3":  (BATCH, BASE,      IMG_H,       IMG_W),
        "dec3": (BATCH, BASE,      IMG_H,       IMG_W),
    }
    for k, v in expected.items():
        assert shape_log[k] == v, f"{k}: got {shape_log[k]}, expected {v}"

    print("✅ ImprovedUNet passes all shape checks!")
    for k, v in shape_log.items():
        print(f"{k:5s}: {v}")

if __name__ == "__main__":
    check_improved_unet_shapes()


✅ ImprovedUNet passes all shape checks!
enc1 : (4, 64, 32, 32)
enc2 : (4, 128, 16, 16)
enc3 : (4, 256, 8, 8)
enc4 : (4, 512, 4, 4)
up1  : (4, 256, 8, 8)
dec1 : (4, 256, 8, 8)
up2  : (4, 128, 16, 16)
dec2 : (4, 128, 16, 16)
up3  : (4, 64, 32, 32)
dec3 : (4, 64, 32, 32)
out  : (4, 1, 32, 32)


In [17]:
def ddpm_schedules(beta1, beta2, T):
    """
    Precomputes all noise scheduling terms needed for training and sampling
    from a denoising diffusion probabilistic model
    Uses a sequence of gradually increasing noise level over T timesteps
    beta1: starting noise level, O(1e-4)
    beta2: final noise level, O(0.02)
    T: number of time steps
    """
    assert beta1 < beta2 < 1.0, "beta1 and beta2 must be in (0, 1)"

    beta_t = (beta2 - beta1) * torch.arange(0, T + 1, dtype=torch.float32) / T + beta1 # noise variance schedule (for every time t in T)
    sqrt_beta_t = torch.sqrt(beta_t)
    alpha_t = 1 - beta_t
    log_alpha_t = torch.log(alpha_t)
    alphabar_t = torch.cumsum(log_alpha_t, dim=0).exp()

    sqrtab = torch.sqrt(alphabar_t)
    oneover_sqrta = 1 / torch.sqrt(alpha_t)

    sqrtmab = torch.sqrt(1 - alphabar_t)
    mab_over_sqrtmab_inv = (1 - alpha_t) / sqrtmab

    # dictionary of schedule terms
    return {
        "alpha_t": alpha_t,  # \alpha_t , signal retention at time step t
        "oneover_sqrta": oneover_sqrta,  # 1/\sqrt{\alpha_t}
        "sqrt_beta_t": sqrt_beta_t,  # \sqrt{\beta_t} , noise scaling factor
        "alphabar_t": alphabar_t,  # \bar{\alpha_t} , cumulative signal retention
        "sqrtab": sqrtab,  # \sqrt{\bar{\alpha_t}} , scales clean image during noise
        "sqrtmab": sqrtmab,  # \sqrt{1-\bar{\alpha_t}} , noise strength
        "mab_over_sqrtmab": mab_over_sqrtmab_inv,  # (1-\alpha_t)/\sqrt{1-\bar{\alpha_t}} , for reverse diffusion
    }


In [23]:

class DDPM1(nn.Module):
    """
    Denoising Diffusion Probabilistic Model
    """
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        """
        betas: tuple (beta1, beta2) for linear noise schedule
        n_T: total number of diffusion steps (e.g. 1000)
        drop_prob: probability of dropping conditioning (for classifier free guidance)
        """
        super(DDPM, self).__init__()
        self.nn_model = nn_model.to(device)

        # register_buffer allows accessing dictionary produced by ddpm_schedules
        # e.g. can access self.sqrtab later
        for k, v in ddpm_schedules(betas[0], betas[1], n_T).items():
            self.register_buffer(k, v)

        self.n_T = n_T
        self.device = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    def forward(self, x, c):
        """
        x: clean image tensor, size (batchsize, 1, 32, 32)
        c: conditional vector, size (batchsize, 32)
        this method is used in training, so samples t and noise randomly
        """

        # t ~ Uniform(0, n_T)
        # sample a random timestep t for each item in the batch, 
        # determines how much noise to add
        _ts = torch.randint(1, self.n_T+1, (x.size(0),)).to(self.device)
        noise = torch.randn_like(x)  # eps ~ N(0, 1)

        # Generates noising image x_t from clean image x
        x_t = (
            self.sqrtab[_ts, None, None, None] * x
            + self.sqrtmab[_ts, None, None, None] * noise
        )  # This is the x_t, which is sqrt(alphabar) x_0 + sqrt(1-alphabar) * eps
        # We should predict the "error term" from this x_t. Loss is what we return.

        # dropout context with some probability
        # context_mask = torch.bernoulli(
        #     torch.zeros_like(c)+self.drop_prob).to(self.device)

        # return MSE between added noise, and our predicted noise
        # runs x_t, c, t, and context_mask through model, compares predicted noise
        # with actual noise using MSE
        # return self.loss_mse(noise, self.nn_model(x_t, c, _ts / self.n_T, context_mask))
        return self.loss_mse(noise, self.nn_model(x_t, c, _ts / self.n_T))

    def sample(self, n_sample, size, device, c_i, guide_w=0.0):
        """
        n_sample: number of images to generate
        size: shape of each image, [1,32,32]
        guid_w: guidance strength, 0=no guidance, >0=stronger conditioning (what we want)
        """
        # we follow the guidance sampling scheme described in 'Classifier-Free Diffusion Guidance'
        # to make the fwd passes efficient, we concat two versions of the dataset,
        # one with context_mask=0 and the other context_mask=1
        # we then mix the outputs with the guidance scale, w
        # where w>0 means more guidance

        # x_T ~ N(0, 1), sample initial noise
        x_i = torch.randn(n_sample, *size).to(device)  # start from pure noise
        # context for us just cycles throught the mnist labels

        x_i_store = []  # keep track of generated steps in case want to plot something
        print()
        # Iterate over timesteps in revers (from noise -> image)
        for i in range(self.n_T, 0, -1):
            print(f'sampling timestep {i}', end='\r')
            t_is = torch.tensor([i / self.n_T], device=device).repeat(n_sample, 1)

            z = torch.randn(n_sample, *size).to(device) if i > 1 else 0 # add noise at all steps except final one

            # predict the noise using both conditioned and unconditioned branches
            eps = self.nn_model(x_i, c_i, t_is)
            # apply classifier-free guidance formula: ϵ = (1 + w)⋅ϵ_cond − w⋅ϵ_uncond

            x_i = (
                self.oneover_sqrta[i] * (x_i - eps * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z
            )
            # save frames for visualization every 20 steps and near the end
            if i % 20 == 0 or i == self.n_T or i < 8:
                x_i_store.append(x_i.detach().cpu().numpy())

        # returns final denoised image x_i and intermediate steps x_i_store
        x_i_store = np.array(x_i_store)
        return x_i, x_i_store

model = ImprovedUNet(in_channels=IN_CH,
                         base=BASE,
                         cond_dim=COND,
                         time_dim=TIME)

ddpm = DDPM(model, betas=(1e-4, 0.02), n_T=10, device='cpu')
x = torch.zeros(4, 1, 32, 32)
c = torch.zeros(4, 8)
loss = ddpm.forward(x, c)
print(loss)

samples, history = ddpm.sample(n_sample=4, size=(1, 32, 32), device='cpu', c_i=c, guide_w=2.0)
print(samples)

tensor(1.2888, grad_fn=<MseLossBackward0>)

tensor([[[[ 0.7490,  0.1864, -1.8164,  ..., -0.9238, -1.6854, -0.2645],
          [-1.6243,  0.3233, -1.4727,  ..., -0.6552, -1.2000, -0.4990],
          [-0.4349, -3.7138, -0.0794,  ..., -0.6074, -1.1714, -0.9760],
          ...,
          [-0.4079, -0.7065,  0.6185,  ..., -0.1599, -1.3296, -0.5091],
          [-0.7168,  0.7405, -1.1846,  ...,  0.0113, -0.6360, -0.1310],
          [ 0.3096, -0.0251, -0.0622,  ...,  1.1167, -0.8104, -0.6973]]],


        [[[ 2.1549,  1.4384, -0.8365,  ...,  0.1115, -0.1714,  1.6511],
          [-1.7610,  1.0142,  2.1732,  ...,  0.7662, -0.2456,  1.4293],
          [-2.2818,  0.1766, -0.4306,  ..., -1.2598,  0.9608,  1.4247],
          ...,
          [-1.4252,  0.0084, -0.7945,  ..., -1.7581,  1.0193, -1.1229],
          [-1.9239,  1.5770,  0.0562,  ..., -0.5132,  2.3039, -0.0583],
          [-0.9511, -0.8411, -0.7930,  ...,  1.5845, -0.5307,  0.4375]]],


        [[[ 1.1428, -1.5110,  0.2729,  ..., -0.3807,  

In [30]:
class DDPM(nn.Module):
    """
    Denoising Diffusion Probabilistic Model with Classifier-Free Guidance
    --------------------------------------------------------------------
    * `nn_model(x, c, t, context_mask)` must accept the extra boolean mask.
      When mask==1 the model should ignore / zero the conditioning.
    """
    def __init__(self, nn_model, betas, n_T, device, drop_prob=0.1):
        super().__init__()
        self.nn_model  = nn_model.to(device)
        for k, v in ddpm_schedules(*betas, n_T).items():
            self.register_buffer(k, v)

        self.n_T      = n_T
        self.device   = device
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    def forward(self, x, c):
        """
        x : [B, 1, 32, 32] clean image
        c : [B, cond_dim]  conditioning vector
        """
        B = x.size(0)
        t  = torch.randint(1, self.n_T + 1, (B,), device=self.device)
        eps = torch.randn_like(x)          # noise ~ N(0,1)

        x_t = self.sqrtab[t, None, None, None] * x \
            + self.sqrtmab[t, None, None, None] * eps

        # sample mask: 1 → DROP conditioning, 0 → keep
        context_mask = torch.bernoulli(
            torch.full((B, 1), self.drop_prob, device=self.device)
        )

        eps_pred = self.nn_model(x_t, c, t / self.n_T, context_mask)
        return self.loss_mse(eps, eps_pred)

    @torch.no_grad()
    def sample(self, n_sample, size, device, c_i, guide_w=0.0):
        """
        guide_w = 0   → unconditional
        guide_w = 1   → full cond − uncond blend (as in paper)
        guide_w > 1   → stronger conditioning
        """
        x = torch.randn(n_sample, *size, device=device)  # x_T
        store = []

        for i in range(self.n_T, 0, -1):
            t = torch.full((n_sample, 1), i / self.n_T, device=device)

            # ---------- predict noise with and without context ----------
            eps_cond  = self.nn_model(x,  c_i, t, torch.zeros_like(t))  # mask=0
            eps_uncond= self.nn_model(x,  torch.zeros_like(c_i), t,
                                      torch.ones_like(t))              # mask=1

            # guidance:  ε = ε_u  + w (ε_c - ε_u)
            eps = eps_uncond + guide_w * (eps_cond - eps_uncond)

            z = torch.randn_like(x) if i > 1 else 0
            x = ( self.oneover_sqrta[i] *
                  (x - eps * self.mab_over_sqrtmab[i])
                + self.sqrt_beta_t[i] * z )

            if i % 20 == 0 or i == self.n_T or i < 8:
                store.append(x.cpu().numpy())

        return x, np.array(store)

model = ImprovedUNet(in_channels=IN_CH,
                         base=BASE,
                         cond_dim=COND,
                         time_dim=TIME)
                         
ddpm = DDPM(model, betas=(1e-4, 0.02), n_T=10, device='cpu')
x = torch.zeros(4, 1, 32, 32)
c = torch.zeros(4, 8)
loss = ddpm.forward(x, c)
print(loss)

samples, history = ddpm.sample(n_sample=4, size=(1, 32, 32), device='cpu', c_i=c, guide_w=2.0)
print(samples)

tensor(0.9767, grad_fn=<MseLossBackward0>)
tensor([[[[ 1.2465,  0.5983, -0.1039,  ...,  0.8731,  0.6850, -0.7252],
          [-0.8696,  0.1369,  0.4597,  ..., -1.7084,  0.3993,  0.4941],
          [ 0.7422,  0.8448, -0.0973,  ...,  0.8656, -0.7066,  0.5077],
          ...,
          [ 1.6532,  1.1372, -0.0326,  ...,  0.2960,  0.2950,  0.1078],
          [-0.6946,  0.1465, -0.8254,  ...,  0.5641, -0.3272,  0.6563],
          [ 0.1115,  0.0558, -0.3325,  ..., -0.0660,  0.6982,  0.5768]]],


        [[[-0.3904,  2.6457, -1.5559,  ..., -0.7261, -1.2328, -0.6344],
          [ 1.9137, -0.3682,  0.1587,  ..., -1.5060,  1.5763,  0.5068],
          [-0.4023, -0.2063, -0.4181,  ...,  0.3736,  0.1288, -0.7143],
          ...,
          [ 0.3578, -1.1167, -0.1847,  ...,  0.1585,  0.4195,  0.2662],
          [-1.8121, -0.8575, -0.6169,  ..., -1.3764,  0.2201,  0.9502],
          [-0.8140,  0.9765,  2.5293,  ..., -0.2012,  1.3961,  0.8212]]],


        [[[-0.3560, -0.8586, -0.1638,  ...,  2.3306, -0

In [38]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision.utils import make_grid, save_image
from tqdm import tqdm
from matplotlib.animation import FuncAnimation, PillowWriter

# ────────────────────────────────────────────────────────────────
def train_waveguide(full_dataset):
    # ─── Hyper-params ────────────────────────────────────────────
    N_EPOCHS  = 50
    BATCH     = 256 if torch.cuda.is_available() else 64
    N_T       = 1_000
    DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
    COND_DIM  = 8
    BASE      = 64
    LR        = 5e-5
    SAVE_DIR  = './data/diffusion_v4/'
    os.makedirs(SAVE_DIR, exist_ok=True)
    CKPT_PATH = os.path.join(SAVE_DIR, "ddpm_latest.pth")
    CURVE_PNG = os.path.join(SAVE_DIR, "loss_curve.png")

    # ─── Train / Test split (80 % / 20 %) ───────────────────────
    n_total   = len(full_dataset)
    n_train   = int(0.8 * n_total)
    n_val     = n_total - n_train
    train_ds, val_ds = random_split(full_dataset,
                                    [n_train, n_val],
                                    generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=BATCH,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH,
                              shuffle=False, num_workers=4, pin_memory=True)

    # ─── Model & Optimiser ──────────────────────────────────────
    model = ImprovedUNet(1, BASE, COND_DIM, 128).to(DEVICE)
    ddpm  = DDPM(nn_model=model,
                 betas=(1e-4, 0.03),
                 n_T=N_T, device=DEVICE,
                 drop_prob=0.1).to(DEVICE)
    optim = torch.optim.Adam(ddpm.parameters(), lr=LR)

    train_losses, val_losses = [], []

    # ───────────────────── EPOCH LOOP ───────────────────────────
    for ep in range(N_EPOCHS):
        # ─── Train ──────────────────────────────────────────────
        ddpm.train()
        optim.param_groups[0]['lr'] = LR * (1 - ep / N_EPOCHS)
        running = 0.0
        for cond, params, imgs in tqdm(train_loader,
                                       desc=f"Epoch {ep+1}/{N_EPOCHS}"):
            imgs, cond = imgs.to(DEVICE), cond.to(DEVICE)
            optim.zero_grad()
            loss = ddpm(imgs, cond)
            loss.backward()
            optim.step()
            running += loss.item() * imgs.size(0)
        train_avg = running / len(train_loader.dataset)
        train_losses.append(train_avg)

        # ─── Validation ─────────────────────────────────────────
        ddpm.eval()
        running_val = 0.0
        with torch.no_grad():
            for cond, params, imgs in tqdm(val_loader):
                imgs, cond = imgs.to(DEVICE), cond.to(DEVICE)
                val_loss = ddpm(imgs, cond)
                running_val += val_loss.item() * imgs.size(0)
        val_avg = running_val / len(val_loader.dataset)
        val_losses.append(val_avg)

        print(f"Epoch {ep+1}: train {train_avg:.6f} | val {val_avg:.6f}")

        # ─── Save latest model (overwrite) ──────────────────────
        torch.save(ddpm.state_dict(), CKPT_PATH)

        # ─── Update & save loss curve ───────────────────────────
        plt.figure(figsize=(8,5))
        plt.plot(train_losses, label='train')
        plt.plot(val_losses,   label='val')
        plt.title("DDPM loss"); plt.xlabel("epoch"); plt.ylabel("MSE")
        plt.legend(); plt.grid()
        plt.tight_layout();   # avoid cut-off
        plt.savefig(CURVE_PNG)
        plt.close()

    print(f"Finished. Final checkpoint at {CKPT_PATH}")


Tasks to complete: completely rework training function for my project
Maybe start with just generating the waveguides, then move to generating waveguides and parameters

In [39]:
from waveguide_dataset import WaveguideDataset
dataset = WaveguideDataset('train_test_split.h5')

if __name__ == "__main__":
    train_waveguide(dataset)


Epoch 1/50:   0%|          | 0/11202 [00:00<?, ?it/s]Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/samueldowd/.pyenv/versions/3.11.1/lib/python3.11/multiprocessing/spawn.py", line 120, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/samueldowd/.pyenv/versions/3.11.1/lib/python3.11/multiprocessing/spawn.py", line 130, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/samueldowd/.pyenv/versions/3.11.1/lib/python3.11/site-packages/torch/__init__.py", line 2108, in <module>
    from torch import _VF as _VF, functional as functional  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/samueldowd/.pyenv/versions/3.11.1/lib/python3.11/site-packages/torch/functional.py", line 7, in <module>
    import torch.nn.functional as F
Epoch 1/50:   0%|          | 0/11202 [00:03<?, ?it/s]dule>n3.1

KeyboardInterrupt: 